# 01 - Feature Engineering with Feature Store

This notebook demonstrates Snowflake's Feature Store for centralized feature management.

**Key Concepts**:
- **Entity**: A business object (e.g., customer) that features describe
- **FeatureView**: A set of feature definitions backed by a Dynamic Table
- **Point-in-time lookups**: Retrieve feature values as of a specific timestamp
- **Automated refresh**: Features update on a configurable schedule

**Prerequisites**: Run `00_data_setup.ipynb` first.

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()
session.sql("USE DATABASE MLOPS_DEMO_DB").collect()
session.sql("USE SCHEMA FEATURES").collect()
print(f"Connected: {session.get_current_user()} | {session.get_current_role()}")

In [ ]:
from snowflake.ml.feature_store import (
    FeatureStore,
    FeatureView,
    Entity,
    CreationMode
)

# Initialize Feature Store
fs = FeatureStore(
    session=session,
    database="MLOPS_DEMO_DB",
    name="FEATURES",
    default_warehouse="MLOPS_DEMO_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)
print("Feature Store initialized.")

In [ ]:
# Define Entity - the business object that features describe
customer_entity = Entity(
    name="CUSTOMER",
    join_keys=["CUSTOMER_ID"]
)
fs.register_entity(customer_entity)
print("Entity 'CUSTOMER' registered.")
print(f"\nAll entities: {fs.list_entities().to_pandas()}")

In [ ]:
# Build feature engineering logic as a Snowpark DataFrame
# These derived features improve model performance
raw_df = session.table("MLOPS_DEMO_DB.RAW.CUSTOMERS")

feature_df = raw_df.select(
    F.col("CUSTOMER_ID"),
    F.col("CREDIT_SCORE"),
    F.col("AGE"),
    F.col("TENURE"),
    F.col("BALANCE"),
    F.col("NUM_OF_PRODUCTS"),
    F.col("HAS_CR_CARD"),
    F.col("IS_ACTIVE_MEMBER"),
    F.col("ESTIMATED_SALARY"),
    # Derived features
    (F.col("BALANCE") / F.iff(F.col("NUM_OF_PRODUCTS") == 0, F.lit(1), F.col("NUM_OF_PRODUCTS"))).alias("BALANCE_PER_PRODUCT"),
    (F.col("ESTIMATED_SALARY") / F.iff(F.col("BALANCE") == 0, F.lit(1), F.col("BALANCE"))).alias("SALARY_BALANCE_RATIO"),
    F.iff(
        (F.col("BALANCE") > 100000) & (F.col("NUM_OF_PRODUCTS") >= 2),
        F.lit(1), F.lit(0)
    ).alias("IS_HIGH_VALUE")
)

print(f"Feature DataFrame columns: {feature_df.columns}")
feature_df.show(5)

In [ ]:
# Register as a FeatureView - backed by a Dynamic Table with automated refresh
customer_fv = FeatureView(
    name="CUSTOMER_FEATURES",
    entities=[customer_entity],
    feature_df=feature_df,
    refresh_freq="1 hour",
    desc="Customer behavior features for churn prediction"
)

customer_fv = fs.register_feature_view(
    feature_view=customer_fv,
    version="v1",
    override=True
)
print(f"FeatureView registered: {customer_fv.name} (version {customer_fv.version})")

In [ ]:
# List all feature views
fv_list = fs.list_feature_views().to_pandas()
print("Registered Feature Views:")
print(fv_list[["NAME", "VERSION", "DESC"]].to_string(index=False))

In [ ]:
# Retrieve features using a spine DataFrame (point-in-time lookup)
# The spine defines which entities and timestamps to look up
spine_df = session.create_dataframe(
    [[1], [2], [3], [4], [5]],
    schema=["CUSTOMER_ID"]
)

training_data = fs.retrieve_feature_values(
    spine_df=spine_df,
    features=[customer_fv]
)
print("Retrieved feature values for 5 customers:")
training_data.to_pandas()

## What to Show in SnowSight

Navigate to **AI/ML > Feature Store** to see:
- Registered entities (CUSTOMER)
- Feature views (CUSTOMER_FEATURES) with version history
- Feature definitions and refresh status
- The underlying Dynamic Table that powers automated refresh

Also check **Data > Databases > MLOPS_DEMO_DB > FEATURES** to see the Dynamic Table backing the FeatureView.

---

**Next**: `02_model_training_xgboost.ipynb` - Train an XGBoost model using these features